# Ejercicio 2. Autómata celular probabilístico: incendio forestal

Se modela un incendio forestal mediante un autómata celular 2D con vecindad de Moore. Los estados son: 0 = vacío, 1 = árbol, 2 = árbol en llamas y 3 = árbol quemado.

## Parámetros y configuración inicial

La simulación utiliza una cuadrícula de 80×80 celdas y una actualización síncrona. La probabilidad de ignición aumenta con el número de vecinos en llamas.

## Simulación y resultados

Se muestran la evolución del bosque, el estado final y el número de celdas de cada estado.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Parámetros
N = 80
pasos = 80
densidad_arboles = 0.65
p_ignicion = 0.35
p_viento = 0.15
semilla = 10

np.random.seed(semilla)

# Estados: 0 vacío, 1 árbol, 2 fuego, 3 quemado
bosque = np.zeros((N, N), dtype=int)
bosque[np.random.random((N, N)) < densidad_arboles] = 1

# Cinco focos iniciales
posiciones = np.argwhere(bosque == 1)
seleccion = posiciones[np.random.choice(len(posiciones), 5, replace=False)]
for i, j in seleccion:
    bosque[i, j] = 2

def vecinos_moore(i, j, matriz):
    vecinos = []
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            if di == 0 and dj == 0:
                continue
            ni, nj = i + di, j + dj
            if 0 <= ni < matriz.shape[0] and 0 <= nj < matriz.shape[1]:
                vecinos.append(matriz[ni, nj])
    return vecinos

def actualizar(bosque):
    nuevo = bosque.copy()
    for i in range(N):
        for j in range(N):
            if bosque[i, j] == 2:
                nuevo[i, j] = 3
            elif bosque[i, j] == 1:
                vecinos = vecinos_moore(i, j, bosque)
                n_fuego = vecinos.count(2)
                if n_fuego > 0:
                    prob = 1 - (1 - p_ignicion)**n_fuego
                    prob = min(1.0, prob + p_viento)
                    if np.random.random() < prob:
                        nuevo[i, j] = 2
    return nuevo


In [ ]:
# Ejecutar la simulación y guardar algunos estados
historial = [bosque.copy()]
for _ in range(pasos):
    bosque = actualizar(bosque)
    historial.append(bosque.copy())

print("Resultados finales")
print("Árboles vivos:", np.sum(bosque == 1))
print("Árboles en llamas:", np.sum(bosque == 2))
print("Árboles quemados:", np.sum(bosque == 3))
print("Celdas vacías:", np.sum(bosque == 0))


In [ ]:
# Visualizar estados representativos
indices = [0, 5, 10, 20, 40, 80]
cmap = ListedColormap(["white", "green", "red", "black"])

for t in indices:
    plt.figure(figsize=(6, 6))
    plt.imshow(historial[t], cmap=cmap, interpolation="nearest")
    plt.title(f"Estado del bosque, ciclo {t}")
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.show()


In [ ]:
# Evolución del número de árboles vivos, en llamas y quemados
vivos = [np.sum(x == 1) for x in historial]
fuego = [np.sum(x == 2) for x in historial]
quemados = [np.sum(x == 3) for x in historial]

plt.figure(figsize=(9, 5))
plt.plot(vivos, label="Árboles vivos")
plt.plot(fuego, label="En llamas")
plt.plot(quemados, label="Quemados")
plt.xlabel("Ciclo")
plt.ylabel("Número de celdas")
plt.title("Evolución del incendio forestal")
plt.legend()
plt.grid()
plt.show()
